# Real-World — Correlations Along a Production Line (MPS Analogy)

We use an entangled chain model (MPS) to sample binary pass/fail states along a line
and compute nearest-neighbor correlation, which can help reason about defect propagation.

In [ ]:
import numpy as np
from quantum_hybrid_system import MatrixProductState

mps = MatrixProductState(num_qubits=32, bond_dim=8)

# Canonicalize for stable sampling
mps.canonicalize_left_to_right()
mps.canonicalize_right_to_left()

# Sample many synthetic "runs" of the line
shots = 2000
samples = mps.sweep_sample(num_shots=shots)  # returns list of bitstrings (as ints or arrays depending on impl)

# Convert to bit arrays of length n
n = 32
bits = np.zeros((shots, n), dtype=int)
for i, s in enumerate(samples):
    # Allow both int or sequence encodings
    if isinstance(s, int):
        for j in range(n):
            bits[i, j] = (s >> j) & 1
    else:
        arr = np.array(s, dtype=int)
        if arr.size == n:
            bits[i] = arr
        else:
            # fallback: take lowest n bits
            val = int(s)
            for j in range(n):
                bits[i, j] = (val >> j) & 1

# Correlation between neighbors
corr = []
for j in range(n-1):
    x = bits[:, j]
    y = bits[:, j+1]
    c = ( (x==y).mean() - (x!=y).mean() )
    corr.append(c)

print("Mean nearest-neighbor correlation:", float(np.mean(corr)))
print("Max/Min correlation:", float(np.max(corr)), float(np.min(corr)))